<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《Build a Large Language Model From Scratch》</a> 一书的补充代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 从零实现字节对编码（BPE）分词器——简化版（Byte Pair Encoding (BPE) Tokenizer From Scratch -- Simple）

- 这是一个独立的 notebook，从零实现了流行的字节对编码（BPE）分词算法（GPT-2 到 GPT-4、Llama 3 等模型都使用了它），目的是教学
- 关于分词的目的，可以参考 [第 2 章](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb)；这里的代码是讲解 BPE 算法的 bonus 内容
- OpenAI 为训练原始 GPT 模型实现的原始 BPE 分词器可以在 [这里](https://github.com/openai/gpt-2/blob/master/src/encoder.py) 找到
- BPE 算法最早在 1994 年的论文 "[A New Algorithm for Data Compression](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)" 中描述，作者 Philip Gage
- 大多数项目（包括 Llama 3）目前都使用 OpenAI 开源的 [tiktoken 库](https://github.com/openai/tiktoken)，因为它的计算性能更好；它支持加载预训练的 GPT-2 和 GPT-4 分词器（Llama 3 模型也是用 GPT-4 分词器训练的）
- 上面这些实现和本 notebook 中我的实现的区别在于，我这里还包含了一个训练分词器的函数（出于教学目的）
- 还有个叫 [minBPE](https://github.com/karpathy/minbpe) 的实现也支持训练，可能性能更好（我这里的实现侧重教学）；和 `minBPE` 不同，我的实现额外支持加载 OpenAI 原始分词器的词表和合并规则

**这是一个非常朴素的实现，仅用于教学。[bpe-from-scratch.ipynb](bpe-from-scratch.ipynb) notebook 中给出了一个更复杂（但也更难读）的实现，它的行为与 tiktoken 一致。**

&nbsp;
# 1. 字节对编码（BPE）背后的核心思想（The main idea behind byte pair encoding (BPE)）

- BPE 的主要思路是把文本转换为整数表示（token ID），用于 LLM 训练（参见 [第 2 章](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb)）

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/bpe-overview.webp" width="600px">

&nbsp;
## 1.1 比特与字节（Bits and bytes）

- 在讲 BPE 算法之前，先介绍一下字节的概念
- 把文本转换成一个 byte 数组（BPE 中的 "byte" 指的就是这个意思）：

In [1]:
text = "This is some text"
byte_ary = bytearray(text, "utf-8")
print(byte_ary)

bytearray(b'This is some text')


- 当我们对 `bytearray` 对象调用 `list()` 时，每个字节会被当作单独的元素，结果是一个整数列表，对应各字节的取值：

In [2]:
ids = list(byte_ary)
print(ids)

[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]


- 这本来也是一种把文本转成 token ID 的有效方式，正是 LLM embedding 层所需要的
- 但这种做法的问题在于：每个字符都对应一个 ID（一段很短的文本就会产生大量 ID！）
- 也就是说，对一段 17 个字符的输入文本，我们得用 17 个 token ID 作为 LLM 的输入：

In [3]:
print("Number of characters:", len(text))
print("Number of token IDs:", len(ids))

Number of characters: 17
Number of token IDs: 17


- 如果你之前用过 LLM，你可能知道 BPE 分词器的词表中，每个 token ID 对应的是整个词或子词，而不是单个字符
- 例如，GPT-2 分词器把同样的文本（"This is some text"）只切成 4 个而不是 17 个 token：`1212, 318, 617, 2420`
- 你可以通过交互式 [tiktoken app](https://tiktokenizer.vercel.app/?model=gpt2) 或 [tiktoken 库](https://github.com/openai/tiktoken) 验证一下：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/tiktokenizer.webp" width="600px">

```python
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")
# prints [1212, 318, 617, 2420]
```

- 因为一个字节由 8 个比特组成，所以一个字节共有 2<sup>8</sup> = 256 种可能的取值，范围从 0 到 255
- 可以通过运行 `bytearray(range(0, 257))` 来验证，它会提示 `ValueError: byte must be in range(0, 256)`
- BPE 分词器通常把这 256 个值作为前 256 个单字符 token；可以通过下面代码直观验证：

```python
import tiktoken
gpt2_tokenizer = tiktoken.get_encoding("gpt2")

for i in range(300):
    decoded = gpt2_tokenizer.decode([i])
    print(f"{i}: {decoded}")
"""
prints:
0: !
1: "
2: #
...
255: �  # <---- 单字符 token 到这里为止
256:  t
257:  a
...
298: ent
299:  n
"""
```

- 注意上面 256 和 257 不是单字符，而是双字符（一个空白 + 一个字母），这是原始 GPT-2 BPE 分词器的一个小缺点（GPT-4 分词器已经改进了）

&nbsp;
## 1.2 构建词表（Building the vocabulary）

- BPE 分词算法的目标是构建一个常用子词的词表，比如 `298: ent`（出现在 *entangle, entertain, enter, entrance, entity, ...* 等词中），甚至是完整的词，如

```
318: is
617: some
1212: This
2420: text
```

- BPE 算法最早在 1994 年的论文 "[A New Algorithm for Data Compression](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)" 中描述，作者 Philip Gage
- 在进入实际代码实现之前，今天 LLM 分词器所用的形式可以概括如下：

&nbsp;
## 1.3 BPE 算法概览（BPE algorithm outline）

**1. 找出高频对（Identify frequent pairs）**
- 每次迭代扫描文本，找到出现最频繁的字节（或字符）对

**2. 替换并记录（Replace and record）**

- 用一个新的占位 ID 替换该对（这个 ID 之前未使用，例如从 0...255 开始的话，第一个占位就是 256）
- 把这个映射记在一个查找表里
- 查找表的大小是一个超参数，也叫"词表大小"（GPT-2 是 50,257）

**3. 重复直到无收益（Repeat until no gains）**

- 不断重复步骤 1 和 2，持续合并出现最频繁的对
- 直到无法再继续压缩为止（例如没有任何对出现超过一次）

**解压（解码）（Decompression (decoding)）**

- 要恢复原始文本，通过查找表把每个 ID 替换回对应的对，反向执行即可




&nbsp;
## 1.4 BPE 算法示例（BPE algorithm example）

### 1.4.1 编码部分的具体示例（步骤 1 和 2）（Concrete example of the encoding part (steps 1 & 2)）

- 假设我们有这样一段文本（训练数据集）`the cat in the hat`，想从中构建 BPE 分词器的词表

**迭代 1**

1. 找出高频对
  - 在这段文本中，"th" 出现了两次（开头一次，第二个 "e" 前面一次）

2. 替换并记录
  - 用一个尚未使用的 token ID（例如 256）替换 "th"
  - 新文本是：`<256>e cat in <256>e hat`
  - 新词表是

```
  0: ...
  ...
  256: "th"
```

**迭代 2**

1. **找出高频对**
   - 在 `<256>e cat in <256>e hat` 中，对 `<256>e` 出现了两次

2. **替换并记录**
   - 用一个尚未使用的新 token ID（比如 `257`）替换 `<256>e`。
   - 新文本是：
     ```
     <257> cat in <257> hat
     ```
   - 更新后的词表是：
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     ```

**迭代 3**

1. **找出高频对**
   - 在 `<257> cat in <257> hat` 中，对 `<257> ` 出现了两次（开头一次，"hat" 前面一次）。

2. **替换并记录**
   - 用一个尚未使用的新 token ID（比如 `258`）替换 `<257> `。
   - 新文本是：
     ```
     <258>cat in <258>hat
     ```
   - 更新后的词表是：
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     258: "<257> "
     ```
     
- 以此类推

&nbsp;
### 1.4.2 解码部分的具体示例（步骤 3）（Concrete example of the decoding part (steps 3)）

- 要恢复原始文本，我们反向执行——按引入顺序的逆序，把每个 token ID 替换回对应的对
- 从最终压缩后的文本开始：`<258>cat in <258>hat`
- 替换 `<258>` → `<257> `：`<257> cat in <257> hat`
- 替换 `<257>` → `<256>e`：`<256>e cat in <256>e hat`
- 替换 `<256>` → "th"：`the cat in the hat`

&nbsp;
## 2. 一个简单的 BPE 实现（A simple BPE implementation）

- 下面用 Python 类实现上述算法，模仿 `tiktoken` 的 Python 用户接口
- 注意：上面的编码部分描述了通过 `train()` 完成原始训练步骤；`encode()` 方法的工作方式类似（只是因为 special token 处理看起来稍微复杂一点）：

1. 把输入文本拆分成单独的字节
2. 反复查找并替换（合并）相邻的 token（对），当它们与已学到的 BPE 合并规则匹配时（按 "rank" 从高到低，即按学到的顺序）
3. 继续合并，直到不能再合并
4. 最终的 token ID 列表就是编码输出

In [ ]:
from collections import Counter, deque
from functools import lru_cache


class BPETokenizerSimple:
    def __init__(self):
        # Maps token_id to token_str (e.g., {11246: "some"})
        self.vocab = {}
        # Maps token_str to token_id (e.g., {"some": 11246})
        self.inverse_vocab = {}
        # Dictionary of BPE merges: {(token_id1, token_id2): merged_token_id}
        self.bpe_merges = {}

    def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
        """
        Train the BPE tokenizer from scratch.

        Args:
            text (str): The training text.
            vocab_size (int): The desired vocabulary size.
            allowed_special (set): A set of special tokens to include.
        """

        # Preprocess: Replace spaces with 'Ġ'
        # Note that Ġ is a particularity of the GPT-2 BPE implementation
        # E.g., "Hello world" might be tokenized as ["Hello", "Ġworld"]
        # (GPT-4 BPE would tokenize it as ["Hello", " world"])
        processed_text = []
        for i, char in enumerate(text):
            if char == " " and i != 0:
                processed_text.append("Ġ")
            if char != " ":
                processed_text.append(char)
        processed_text = "".join(processed_text)

        # Initialize vocab with unique characters, including 'Ġ' if present
        # Start with the first 256 ASCII characters
        unique_chars = [chr(i) for i in range(256)]

        # Extend unique_chars with characters from processed_text that are not already included
        unique_chars.extend(char for char in sorted(set(processed_text)) if char not in unique_chars)

        # Optionally, ensure 'Ġ' is included if it is relevant to your text processing
        if 'Ġ' not in unique_chars:
            unique_chars.append('Ġ')

        # Now create the vocab and inverse vocab dictionaries
        self.vocab = {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}

        # Add allowed special tokens
        if allowed_special:
            for token in allowed_special:
                if token not in self.inverse_vocab:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id

        # Tokenize the processed_text into token IDs
        token_ids = [self.inverse_vocab[char] for char in processed_text]

        # BPE steps 1-3: Repeatedly find and replace frequent pairs
        for new_id in range(len(self.vocab), vocab_size):
            if len(token_ids) < 2:
                break
            pair_id = self.find_freq_pair(token_ids, mode="most")
            if pair_id is None:  # No more pairs to merge. Stopping training.
                break
            
            updated = self.replace_pair(token_ids, pair_id, new_id)
            if updated == token_ids:
                break

            token_ids = updated
            self.bpe_merges[pair_id] = new_id

        # Build the vocabulary with merged tokens
        for (p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id

    def encode(self, text):
        """
        Encode the input text into a list of token IDs.

        Args:
            text (str): The text to encode.

        Returns:
            List[int]: The list of token IDs.
        """
        tokens = []
        # Split text into tokens, keeping newlines intact
        words = text.replace("\n", " \n ").split()  # Ensure '\n' is treated as a separate token

        for i, word in enumerate(words):
            if i > 0 and not word.startswith("\n"):
                tokens.append("Ġ" + word)  # Add 'Ġ' to words that follow a space or newline
            else:
                tokens.append(word)  # Handle first word or standalone '\n'

        token_ids = []
        for token in tokens:
            if token in self.inverse_vocab:
                # token is contained in the vocabulary as is
                token_id = self.inverse_vocab[token]
                token_ids.append(token_id)
            else:
                # Attempt to handle subword tokenization via BPE
                sub_token_ids = self.tokenize_with_bpe(token)
                token_ids.extend(sub_token_ids)

        return token_ids

    def tokenize_with_bpe(self, token):
        """
        Tokenize a single token using BPE merges.

        Args:
            token (str): The token to tokenize.

        Returns:
            List[int]: The list of token IDs after applying BPE.
        """
        # Tokenize the token into individual characters (as initial token IDs)
        token_ids = [self.inverse_vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"Characters not found in vocab: {missing_chars}")

        can_merge = True
        while can_merge and len(token_ids) > 1:
            can_merge = False
            new_tokens = []
            i = 0
            while i < len(token_ids) - 1:
                pair = (token_ids[i], token_ids[i + 1])
                if pair in self.bpe_merges:
                    merged_token_id = self.bpe_merges[pair]
                    new_tokens.append(merged_token_id)
                    # Uncomment for educational purposes:
                    # print(f"Merged pair {pair} -> {merged_token_id} ('{self.vocab[merged_token_id]}')")
                    i += 2  # Skip the next token as it's merged
                    can_merge = True
                else:
                    new_tokens.append(token_ids[i])
                    i += 1
            if i < len(token_ids):
                new_tokens.append(token_ids[i])
            token_ids = new_tokens

        return token_ids

    def decode(self, token_ids):
        """
        Decode a list of token IDs back into a string.

        Args:
            token_ids (List[int]): The list of token IDs to decode.

        Returns:
            str: The decoded string.
        """
        decoded_string = ""
        for token_id in token_ids:
            if token_id not in self.vocab:
                raise ValueError(f"Token ID {token_id} not found in vocab.")
            token = self.vocab[token_id]
            if token.startswith("Ġ"):
                # Replace 'Ġ' with a space
                decoded_string += " " + token[1:]
            else:
                decoded_string += token
        return decoded_string

    @lru_cache(maxsize=None)
    def get_special_token_id(self, token):
        return self.inverse_vocab.get(token, None)

    @staticmethod
    def find_freq_pair(token_ids, mode="most"):
        if(len(token_ids) < 2):
            return None
        pairs = Counter(zip(token_ids, token_ids[1:]))
        if not pairs:
            return None

        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")

    @staticmethod
    def replace_pair(token_ids, pair_id, new_id):
        dq = deque(token_ids)
        replaced = []

        while dq:
            current = dq.popleft()
            if dq and (current, dq[0]) == pair_id:
                replaced.append(new_id)
                # Remove the 2nd token of the pair, 1st was already removed
                dq.popleft()
            else:
                replaced.append(current)

        return replaced


### 短 token 序列的边界处理（Edge-case handling for short token sequences）

BPE 合并要求相邻的 token 对。
如果 token 序列少于 2 项，就不存在对，此时 `find_freq_pair` 返回 `None`，训练优雅停止。

In [21]:
tok = BPETokenizerSimple()

assert tok.find_freq_pair([]) is None
assert tok.find_freq_pair([42]) is None

tok.train("", vocab_size=270)
tok.train("H", vocab_size=270)
tok.train("He", vocab_size=270)

print("Edge-case checks passed.")

Edge-case checks passed.


- 上面 `BPETokenizerSimple` 类的代码很多，详细讨论超出了本 notebook 的范围，但下一节给出了一个简短的使用概览，让我们能更好地理解这些类方法

## 3. BPE 实现走读（BPE implementation walkthrough）

- 实际使用中，我强烈推荐用 [tiktoken](https://github.com/openai/tiktoken)，因为上面的实现注重可读性和教学目的，而不是性能
- 但用法大致和 tiktoken 相似，只是 tiktoken 没有训练方法
- 我们通过下面几个例子看一下我上面写的 `BPETokenizerSimple` Python 代码是如何工作的（详细的代码讨论超出了本 notebook 的范围）

### 3.1 训练、编码和解码（Training, encoding, and decoding）

- 首先，准备一段示例文本作为我们的训练数据集：

In [5]:
import os
import urllib.request

if not os.path.exists("../01_main-chapter-code/the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "../01_main-chapter-code/the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

with open("../01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f: # added ../01_main-chapter-code/
    text = f.read()

- 接下来，初始化并训练一个词表大小为 1,000 的 BPE 分词器
- 注意：默认情况下词表大小已经是 255（因为前面讨论的字节值），所以我们实际上只"学"了 745 个词表项
- 作为对比，GPT-2 词表是 50,257 个 token，GPT-4 是 100,256 个 token（tiktoken 中的 `cl100k_base`），GPT-4o 用 199,997 个 token（tiktoken 中的 `o200k_base`）；它们的训练集都比我们这里简单的示例文本大得多

In [6]:
tokenizer = BPETokenizerSimple()
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})

- 你可能想查看一下词表的内容（注意这会产生一个很长的列表）

In [7]:
# print(tokenizer.vocab)
print(len(tokenizer.vocab))

1000


- 这个词表是通过约 742 次合并生成的（~ `1000 - len(range(0, 256))`）

In [8]:
print(len(tokenizer.bpe_merges))

742


- 这意味着前 256 项是单字符 token

- 接下来，通过 `encode` 方法使用学到的合并规则对一些文本做编码：

In [9]:
input_text = "Jack embraced beauty through art and life."
token_ids = tokenizer.encode(input_text)
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]


In [10]:
print("Number of characters:", len(input_text))
print("Number of token IDs:", len(token_ids))

Number of characters: 42
Number of token IDs: 20


- 从上面的长度可以看到，42 个字符的句子被编码成了 20 个 token ID，相比按字符/字节的编码方式，输入长度大约被压缩了一半

- 注意 `decode()` 方法会用到词表本身，它可以把 token ID 映射回文本：

In [11]:
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]


In [12]:
print(tokenizer.decode(token_ids))

Jack embraced beauty through art and life.


- 遍历每个 token ID 可以让我们更好地理解 token ID 是如何通过词表解码回文本的：

In [13]:
for token_id in token_ids:
    print(f"{token_id} -> {tokenizer.decode([token_id])}")

424 -> Jack
256 ->  
654 -> em
531 -> br
302 -> ac
311 -> ed
256 ->  
296 -> be
97 -> a
465 -> ut
121 -> y
595 ->  through
841 ->  ar
116 -> t
287 ->  a
466 -> nd
256 ->  
326 -> li
972 -> fe
46 -> .


- 可以看到，大多数 token ID 对应 2 个字符的子词；这是因为训练数据文本非常短，没有那么多重复的词，加上我们用的词表相对较小

- 总结一下，调用 `decode(encode())` 应该能复现任意输入文本：

In [14]:
tokenizer.decode(tokenizer.encode("This is some text."))

'This is some text.'

&nbsp;
# 4. 结论（Conclusion）

- 就这样！这就是 BPE 的工作原理简介，还附带一个用于创建新分词器的训练方法
- 希望这个简短的教程对教学有帮助；如果有疑问，欢迎在 [这里](https://github.com/rasbt/LLMs-from-scratch/discussions/categories/q-a) 发起新的 Discussion


**这是一个非常朴素的实现，仅用于教学。[bpe-from-scratch.ipynb](bpe-from-scratch.ipynb) notebook 中给出了一个更复杂（但也更难读）的实现，它的行为与 tiktoken 一致。**